# GeoRA：全部目標層的初始化檢查

上一個 notebook 只替換第 0 層的 Q。這次將同一方法套到 **28 個 block × 7 種投影 = 196 個線性層**。

這一步回答：初始化後完整模型的輸出變了多少？可訓練的參數是否只有 A/B？保存後能否重新載入？

依 [PRECISION.md](PRECISION.md)，本機全部計算使用 CPU FP32；尚未進行 GPU BF16 訓練或 GRPO。

從上到下執行。兩次 SVD 對每層只做一次，會顯示進度。重新完整執行會重新做初始化。

In [ ]:
import gc
import json
import time
from pathlib import Path

import torch
import transformers
from safetensors.torch import load_file, save_file
from transformers import AutoModelForCausalLM, AutoTokenizer

from geora_layers import (
    GeoRALinear,
    export_adapter_state,
    install_geora,
    iter_target_linears,
    load_geora_state,
)

torch.set_num_threads(4)
project_directory = Path.cwd().resolve()
assert (project_directory / "geora_layers.py").is_file(), "Run from the repository root."
checkpoint_directory = project_directory / 'checkpoints/geora_base'
output_directory = project_directory / 'outputs/geora_full_model_check'
model_repository = 'Qwen/Qwen2.5-1.5B-Instruct'
model_revision = '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'

# Check the local download's recorded revision before loading its weights.
download_record = checkpoint_directory / '.cache/huggingface/download/model.safetensors.metadata'
assert download_record.read_text().splitlines()[0] == model_revision

rank = 16
alpha = 32
rho = 0.2
print('Initialization and forward: CPU FP32')
print('r =', rank, '; alpha =', alpha, '; rho =', rho)

## 1. 先取得原始模型的 logits

原始 checkpoint 保存為 BF16；載入時轉 FP32。先固定同一段輸入，記錄每個位置、整個 vocabulary 的 logits。

後面比較初始化後模型時仍使用這段輸入與 FP32 forward。`eval()` 停用 dropout；`inference_mode()` 表示這次只計算輸出。

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    checkpoint_directory,
    dtype=torch.float32,
    attn_implementation='eager',
    local_files_only=True,
)
model.eval()
model.requires_grad_(False)

tokenizer = AutoTokenizer.from_pretrained(
    checkpoint_directory,
    local_files_only=True,
)
prompt = 'What is 2 + 3? Give a short answer.'
inputs = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': prompt}],
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
    return_dict=True,
)
with torch.inference_mode():
    logits_before = model(**inputs, use_cache=False).logits.clone()

print('Input shape:', tuple(inputs['input_ids'].shape))
print('Logits shape:', tuple(logits_before.shape))

## 2. 確認要替換哪些層

每個 block 的 Q、K、V、O，以及 MLP 的 gate、up、down，共 7 個線性層。`lm_head` 不替換，bias 也不訓練。

如果原權重是 `[out, in]`，新增 A 為 `[r, in]`、B 為 `[out, r]`，可訓練參數數量是 `r × (in + out)`。

In [ ]:
target_shapes = [
    (name, tuple(layer.weight.shape))
    for name, layer in iter_target_linears(model)
]
expected_module_count = 28 * 7
expected_trainable_count = sum(
    rank * (output_size + input_size)
    for name, (output_size, input_size) in target_shapes
)
assert len(target_shapes) == expected_module_count
assert expected_trainable_count == 18_464_768

print('Target linear layers:', len(target_shapes))
print('Trainable A/B parameters:', f'{expected_trainable_count:,}')
print('First block:')
for name, shape in target_shapes[:7]:
    print(' ', name, shape)

## 3. 逐層初始化

`install_geora` 對每一層執行已學過的三步：

1. 兩個 mask 取聯集，得到 `W_geo`。
2. 對 `W_geo` 做 SVD，得到 `B0 @ A0` 的 rank-r 近似。
3. 計算並凍結 `F = W_pre - (alpha/r) * (B0 @ A0)`，只讓 A/B 可訓練。

共用程式放在 [geora_layers.py](geora_layers.py)。這樣不用在 notebook 重複貼 196 次相同操作。每次只保留一層的 SVD 中間結果。

In [ ]:
initialization_started = time.perf_counter()

def show_progress(completed, total, name, statistics):
    if completed % 7 == 0 or completed == total:
        elapsed = time.perf_counter() - initialization_started
        print(f'[{completed}/{total}] {name}; elapsed={elapsed:.1f}s', flush=True)

manifest = install_geora(
    model,
    rank=rank,
    alpha=alpha,
    rho=rho,
    progress_callback=show_progress,
)
initialization_seconds = time.perf_counter() - initialization_started

expected_trainable_names = {
    f"{target['name']}.{factor_name}"
    for target in manifest['target_modules']
    for factor_name in ('A', 'B')
}
actual_trainable_names = {
    name for name, parameter in model.named_parameters()
    if parameter.requires_grad
}
assert actual_trainable_names == expected_trainable_names
assert all(parameter.dtype == torch.float32 for parameter in model.parameters())
trainable_count = sum(
    parameter.numel() for parameter in model.parameters()
    if parameter.requires_grad
)
assert trainable_count == expected_trainable_count

weight_errors = [
    target['statistics']['initial_effective_weight_relative_error']
    for target in manifest['target_modules']
]
print('Only A/B trainable:', len(actual_trainable_names), 'parameter tensors')
print('Largest effective-weight relative error:', max(weight_errors))
print('Initialization time:', f'{initialization_seconds:.1f}s')

## 4. 比較完整模型的 logits

記錄兩個誤差：

- 相對誤差：`norm(logits_initialized - logits_before) / norm(logits_before)`。
- 最大絕對差：所有位置與 vocabulary 中最大的 `abs(logits_initialized - logits_before)`。

數學上初始有效權重等於 `W_pre`；浮點減法與兩條 forward 分支會產生舍入。因此量測實際誤差，不要求與原模型逐 bit 相同。這個固定輸入檢查也不能代表整個 GSM8K 的表現。

In [ ]:
model.eval()
with torch.inference_mode():
    logits_initialized = model(**inputs, use_cache=False).logits.clone()

logits_difference = logits_initialized - logits_before
logits_relative_error = (
    torch.linalg.vector_norm(logits_difference)
    / torch.linalg.vector_norm(logits_before)
).item()
logits_maximum_error = logits_difference.abs().max().item()
assert torch.isfinite(logits_initialized).all()

last_token_before = logits_before[0, -1].argmax().item()
last_token_initialized = logits_initialized[0, -1].argmax().item()
print('Logits relative error:', logits_relative_error)
print('Logits maximum absolute difference:', logits_maximum_error)
print('Last-position argmax token IDs:', last_token_before, last_token_initialized)
print('All logits finite:', True)

## 5. 保存 A/B，以及重建 F 所需的初始 A0/B0

保存原始模型版本、設定與每層的四個因子。原始 checkpoint 已在本機，不重複保存整個凍結模型。

之後從同一份 `W_pre` 與初始 A0/B0 重建 F，再載入當前 A/B。本次尚未訓練，當前 A/B 就是初始值；訓練後兩者會不同。

In [ ]:
output_directory.mkdir(parents=True, exist_ok=True)
adapter_path = output_directory / 'adapter.safetensors'
manifest_path = output_directory / 'manifest.json'
manifest.update({
    'model_repository': model_repository,
    'model_revision': model_revision,
    'original_weight_dtype': 'bfloat16',
    'runtime_device': 'cpu',
    'runtime_dtype': 'float32',
    'torch_threads': torch.get_num_threads(),
    'torch_version': torch.__version__,
    'transformers_version': transformers.__version__,
    'attention_implementation': 'eager',
    'scope': 'all_196_projection_initialization_and_reload_without_training',
})
adapter_state = export_adapter_state(model)
save_file(adapter_state, str(adapter_path))
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n')
del adapter_state

print('Factor tensors saved:', expected_module_count * 4)
print('Adapter checkpoint size:', f'{adapter_path.stat().st_size / 1_000_000:.1f} MB')
print('Saved to:', output_directory)

## 6. 從原始模型重新載入

先釋放現在的完整模型，再重新讀入原始 checkpoint，避免同時保留兩份 FP32 模型。用保存的初始因子重建 F，不重新跑 SVD。

這次比較的是**保存前後的同一個 GeoRA 模型**，與第 4 步「原始模型 vs 初始化模型」的比較不同。相同 CPU 環境與計算順序下應能重現相同輸出。

In [ ]:
del model
gc.collect()

saved_manifest = json.loads(manifest_path.read_text())
assert saved_manifest['model_revision'] == model_revision
assert saved_manifest['model_repository'] == model_repository
model = AutoModelForCausalLM.from_pretrained(
    checkpoint_directory,
    dtype=torch.float32,
    attn_implementation='eager',
    local_files_only=True,
)
saved_state = load_file(str(adapter_path), device='cpu')
load_geora_state(model, saved_state, saved_manifest)
del saved_state
model.eval()

with torch.inference_mode():
    logits_reloaded = model(**inputs, use_cache=False).logits
reload_maximum_error = (logits_reloaded - logits_initialized).abs().max().item()
assert torch.equal(logits_reloaded, logits_initialized)
assert {
    name for name, parameter in model.named_parameters()
    if parameter.requires_grad
} == expected_trainable_names

checks = {
    'scope': 'all_196_projection_initialization_and_reload_without_training',
    'model_repository': model_repository,
    'model_revision': model_revision,
    'target_module_count': expected_module_count,
    'trainable_parameter_count': trainable_count,
    'all_model_parameters_dtype': 'float32',
    'initialization_seconds': initialization_seconds,
    'largest_effective_weight_relative_error': max(weight_errors),
    'logits_relative_error': logits_relative_error,
    'logits_maximum_absolute_difference': logits_maximum_error,
    'last_position_argmax_before': last_token_before,
    'last_position_argmax_initialized': last_token_initialized,
    'reload_logits_maximum_absolute_difference': reload_maximum_error,
    'only_AB_trainable': True,
    'all_logits_finite': True,
    'reload_logits_exactly_equal': True,
    'optimizer_steps': 0,
}
checks_path = output_directory / 'checks.json'
checks_path.write_text(json.dumps(checks, ensure_ascii=False, indent=2) + '\n')
print('Reload logits maximum absolute difference:', reload_maximum_error)
print('Saved measurements:', checks_path)

# Release the full model after the checks to return memory to the local system.
del model
gc.collect()

## 7. 完成後接什麼？

這個 notebook 測量了全模型初始化誤差、可訓練參數清單，以及初始化 checkpoint 的重新載入。

下一步是 **GPU 上的 BF16 forward 與 A/B 單步更新檢查**：凍結部分 BF16、A/B FP32，確認實際 dtype、梯度及凍結權重不變。再接 GSM8K 的短 GRPO 試跑；最後才比較 LoRA 與 GeoRA 的分數。

`geora_layers.py` 是我們依論文寫的本機初始化實作，尚未接入 PEFT、TRL 或 verl。停用 adapter 只會露出 F，不能直接當原始模型的 reference。